In [48]:
import os
os.getcwd()

'/workspaces/music_data_analysis_project/pumpkin_price_analysis_project'

In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 数据读取
df = pd.read_csv('US-pumpkins.csv')

# 查看数据基本信息
print("数据集基本信息：")
df.info()
df.head()

数据集基本信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1757 entries, 0 to 1756
Data columns (total 26 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   City Name        1757 non-null   object 
 1   Type             45 non-null     object 
 2   Package          1757 non-null   object 
 3   Variety          1752 non-null   object 
 4   Sub Variety      296 non-null    object 
 5   Grade            0 non-null      float64
 6   Date             1757 non-null   object 
 7   Low Price        1757 non-null   float64
 8   High Price       1757 non-null   float64
 9   Mostly Low       1654 non-null   float64
 10  Mostly High      1654 non-null   float64
 11  Origin           1754 non-null   object 
 12  Origin District  131 non-null    object 
 13  Item Size        1478 non-null   object 
 14  Color            1141 non-null   object 
 15  Environment      0 non-null      float64
 16  Unit of Sale     162 non-null    object 
 17  Quali

,City Name,Type,Package,Variety,Sub Variety,Grade,Date,Low Price,High Price,Mostly Low,...,Unit of Sale,Quality,Condition,Appearance,Storage,Crop,Repack,Trans Mode,Unnamed: 24,Unnamed: 25
0,BALTIMORE,NaN,24 inch bins,NaN,NaN,NaN,4/29/17,270.0,280.0,270.0,...,NaN,NaN,NaN,NaN,NaN,NaN,E,NaN,NaN,NaN
1,BALTIMORE,NaN,24 inch bins,NaN,NaN,NaN,5/6/17,270.0,280.0,270.0,...,NaN,NaN,NaN,NaN,NaN,NaN,E,NaN,NaN,NaN
2,BALTIMORE,NaN,24 inch bins,HOWDEN TYPE,NaN,NaN,9/24/16,160.0,160.0,160.0,...,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN
3,BALTIMORE,NaN,24 inch bins,HOWDEN TYPE,NaN,NaN,9/24/16,160.0,160.0,160.0,...,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN
4,BALTIMORE,NaN,24 inch bins,HOWDEN TYPE,NaN,NaN,11/5/16,90.0,100.0,90.0,...,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN


In [6]:
# 筛选重要特征
selected_columns = [
    'Date', 'City Name', 'Type', 'Package', 'Variety', 
    'Sub Variety', 'Grade', 'Environment', 'Unit of Sale',
    'Quality', 'Condition', 'Appearance', 'Storage', 
    'Crop', 'Repack', 'Mostly Low', 'Mostly High'
]

df_clean = df[selected_columns].copy()

In [7]:
# 深度缺失值处理流程
# 初始缺失值统计
initial_missing = df_clean.isnull().sum()
print("初始缺失值统计：")
print(initial_missing[initial_missing > 0])

初始缺失值统计：
Type            1712
Variety            5
Sub Variety     1461
Grade           1757
Environment     1757
Unit of Sale    1595
Quality         1757
Condition       1757
Appearance      1757
Storage         1757
Crop            1757
Mostly Low       103
Mostly High      103
dtype: int64


In [8]:
# 数值型列处理（使用中位数填充）
numeric_cols = df_clean.select_dtypes(include='number').columns
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f"填充数值列 {col} 的缺失值，中位数：{median_val}")

填充数值列 Grade 的缺失值，中位数：nan
填充数值列 Environment 的缺失值，中位数：nan
填充数值列 Quality 的缺失值，中位数：nan
填充数值列 Condition 的缺失值，中位数：nan
填充数值列 Appearance 的缺失值，中位数：nan
填充数值列 Storage 的缺失值，中位数：nan
填充数值列 Crop 的缺失值，中位数：nan
填充数值列 Mostly Low 的缺失值，中位数：147.0
填充数值列 Mostly High 的缺失值，中位数：150.0


In [9]:
# 分类列处理（使用'unknown'填充）
object_cols = df_clean.select_dtypes(include='object').columns
for col in object_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna('unknown', inplace=True)
        print(f"填充分类列 {col} 的缺失值，值：'unknown'")

填充分类列 Type 的缺失值，值：'unknown'
填充分类列 Variety 的缺失值，值：'unknown'
填充分类列 Sub Variety 的缺失值，值：'unknown'
填充分类列 Unit of Sale 的缺失值，值：'unknown'


In [10]:
# 日期特征工程
if 'Date' in df_clean.columns:
    try:
        df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce')  # 错误转换设为NaT
        # 检查日期转换是否产生缺失值
        date_missing = df_clean['Date'].isnull().sum()
        if date_missing > 0:
            print(f"日期转换产生 {date_missing} 个缺失值，使用众数日期填充")
            mode_date = df_clean['Date'].mode()[0]
            df_clean['Date'].fillna(mode_date, inplace=True)
        
        df_clean['Month'] = df_clean['Date'].dt.month
        df_clean['DayOfWeek'] = df_clean['Date'].dt.dayofweek
        df_clean = df_clean.drop(['Date'], axis=1)
    except Exception as e:
        print(f"日期处理错误：{e}")
        df_clean = df_clean.drop(['Date'], axis=1)  # 无法处理则丢弃日期列

In [11]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 标签编码处理
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        print(f"编码前 {col} 仍有 {df_clean[col].isnull().sum()} 个缺失值")
    le = LabelEncoder()
    try:
        df_clean[col] = le.fit_transform(df_clean[col])
    except Exception as e:
        print(f"编码列 {col} 时出错：{e}")
        # 处理编码错误，如所有值相同导致的错误
        df_clean[col] = 0  

In [12]:
# 标准化处理
try:
    scaler = StandardScaler()
    numeric_cols = df_clean.select_dtypes(include='number').columns.tolist()
    # 排除目标变量列
    if 'Mostly Low' in numeric_cols:
        numeric_cols.remove('Mostly Low')
    if 'Mostly High' in numeric_cols:
        numeric_cols.remove('Mostly High')
    
    if numeric_cols:  # 确保有列可标准化
        df_clean[numeric_cols] = scaler.fit_transform(df_clean[numeric_cols])
        # 检查标准化后是否有NaN
        std_missing = df_clean[numeric_cols].isnull().sum().sum()
        if std_missing > 0:
            print(f"标准化产生 {std_missing} 个缺失值，使用0填充")
            df_clean[numeric_cols] = df_clean[numeric_cols].fillna(0)
except Exception as e:
    print(f"标准化错误：{e}")

标准化产生 12299 个缺失值，使用0填充


In [13]:
# 最终缺失值检查
final_missing = df_clean.isnull().sum()
final_missing.isnull().sum()

np.int64(0)

In [14]:
from sklearn.feature_selection import SelectKBest, f_regression

# 特征选择
X = df_clean.drop(['Mostly Low', 'Mostly High'], axis=1)
y_low = df_clean['Mostly Low']
y_high = df_clean['Mostly High']

# 使用SelectKBest选择最重要的特征
selector = SelectKBest(score_func=f_regression, k=10)
X_low_selected = selector.fit_transform(X, y_low)
X_high_selected = selector.fit_transform(X, y_high)

# 获取特征得分
low_scores = pd.Series(selector.scores_, index=X.columns)
high_scores = pd.Series(selector.scores_, index=X.columns)

print("\n低价特征重要性:")
print(low_scores.sort_values(ascending=False).head(10))
print("\n高价特征重要性:")
print(high_scores.sort_values(ascending=False).head(10))


低价特征重要性:
Sub Variety     445.164675
Package         253.640453
Variety         119.168504
Month            46.767487
City Name        37.497556
Repack           29.435784
Type              4.668289
DayOfWeek         3.246436
Unit of Sale      2.374435
Grade             0.000000
dtype: float64

高价特征重要性:
Sub Variety     445.164675
Package         253.640453
Variety         119.168504
Month            46.767487
City Name        37.497556
Repack           29.435784
Type              4.668289
DayOfWeek         3.246436
Unit of Sale      2.374435
Grade             0.000000
dtype: float64


In [15]:
from sklearn.model_selection import train_test_split

# 划分数据集
X_low_train, X_low_test, y_low_train, y_low_test = train_test_split(
    X_low_selected, y_low, test_size=0.25, random_state=42
)

X_high_train, X_high_test, y_high_train, y_high_test = train_test_split(
    X_high_selected, y_high, test_size=0.25, random_state=42
)

In [16]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

# 定义模型评估函数
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    print(f"\n===== {model_name} =====")
    # 交叉验证
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    cv_rmse = np.sqrt(-cv_scores.mean())
    
    # 训练模型
    model.fit(X_train, y_train)
    
    # 预测
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # 评估指标
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2 = r2_score(y_test, y_pred_test)
    
    print(f'交叉验证RMSE: {cv_rmse:.2f}')
    print(f'训练集RMSE: {train_rmse:.2f}')
    print(f'测试集RMSE: {test_rmse:.2f}')
    print(f'R² 分数: {r2:.2f}')
    
    # 检查过拟合
    if train_rmse < test_rmse:
        overfit = ((test_rmse - train_rmse) / train_rmse) * 100
        print(f'过拟合程度: {overfit:.2f}%')
    
    return model, y_pred_test, {
        'model_name': model_name,
        'cv_rmse': cv_rmse,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'r2': r2,
        'overfit_percentage': overfit if train_rmse < test_rmse else 0
    }

In [17]:
# 存储模型结果
model_results_low = []
model_results_high = []

In [18]:
from sklearn.linear_model import Ridge

# 线性回归模型（使用Ridge正则化）
model, _, metrics = evaluate_model(
    Ridge(alpha=1.0), 
    X_low_train, X_low_test, y_low_train, y_low_test,
    "低价预测 - Ridge回归"
)
model_results_low.append(metrics)

model, _, metrics = evaluate_model(
    Ridge(alpha=1.0), 
    X_high_train, X_high_test, y_high_train, y_high_test,
    "高价预测 - Ridge回归"
)
model_results_high.append(metrics)


===== 低价预测 - Ridge回归 =====
交叉验证RMSE: 68.51
训练集RMSE: 67.63
测试集RMSE: 73.97
R² 分数: 0.27
过拟合程度: 9.38%

===== 高价预测 - Ridge回归 =====
交叉验证RMSE: 70.06
训练集RMSE: 69.19
测试集RMSE: 75.24
R² 分数: 0.27
过拟合程度: 8.74%


In [19]:
from sklearn.ensemble import RandomForestRegressor

# 随机森林模型
model, _, metrics = evaluate_model(
    RandomForestRegressor(
        n_estimators=150,
        max_depth=8,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42
    ),
    X_low_train, X_low_test, y_low_train, y_low_test,
    "低价预测 - 随机森林"
)
model_results_low.append(metrics)

model, _, metrics = evaluate_model(
    RandomForestRegressor(
        n_estimators=150,
        max_depth=8,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42
    ),
    X_high_train, X_high_test, y_high_train, y_high_test,
    "高价预测 - 随机森林"
)
model_results_high.append(metrics)


===== 低价预测 - 随机森林 =====


交叉验证RMSE: 39.47
训练集RMSE: 33.53
测试集RMSE: 39.89
R² 分数: 0.79
过拟合程度: 18.96%

===== 高价预测 - 随机森林 =====
交叉验证RMSE: 39.79
训练集RMSE: 34.34
测试集RMSE: 40.13
R² 分数: 0.79
过拟合程度: 16.87%


In [22]:
import xgboost as xgb

# XGBoost模型
model, y_low_pred_xgb, metrics = evaluate_model(
    xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        min_child_weight=1,
        gamma=0,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.05,
        reg_lambda=0.05,
        random_state=42
    ),
    X_low_train, X_low_test, y_low_train, y_low_test,
    "低价预测 - XGBoost"
)
model_results_low.append(metrics)

model, y_high_pred_xgb, metrics = evaluate_model(
    xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        min_child_weight=1,
        gamma=0,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.05,
        reg_lambda=0.05,
        random_state=42
    ),
    X_high_train, X_high_test, y_high_train, y_high_test,
    "高价预测 - XGBoost"
)
model_results_high.append(metrics)


===== 低价预测 - XGBoost =====


交叉验证RMSE: 23.56
训练集RMSE: 16.28
测试集RMSE: 18.43
R² 分数: 0.95
过拟合程度: 13.25%

===== 高价预测 - XGBoost =====
交叉验证RMSE: 23.72
训练集RMSE: 16.47
测试集RMSE: 19.89
R² 分数: 0.95
过拟合程度: 20.76%


In [23]:
# 输出最佳模型
best_low_model = max(model_results_low, key=lambda x: x['r2'])
best_high_model = max(model_results_high, key=lambda x: x['r2'])

print("\n===== 最佳模型 =====")
print(f"低价预测最佳模型: {best_low_model['model_name']}, R²分数: {best_low_model['r2']:.4f}")
print(f"高价预测最佳模型: {best_high_model['model_name']}, R²分数: {best_high_model['r2']:.4f}")


===== 最佳模型 =====
低价预测最佳模型: 低价预测 - XGBoost, R²分数: 0.9546
高价预测最佳模型: 高价预测 - XGBoost, R²分数: 0.9489
